# Comment Category Prediction

## Submission By:

- **Name:** Varun Agnihotri
- **Roll Number:** `24f2004142`

## Import all the required libraries

In [ ]:
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

## Config

In [ ]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("datasets/")

## Data loading

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

In [ ]:
for _name, _df in [
    ("train", train),
    ("test", test),
    ("sample_sub", sample_sub),
]:
    print(f" {_name} ".center(50, "="))
    print("\nDataset Info:")
    print(_df.info())
    print("\nDataset Description:")
    print(_df.describe(include="all"))
    # print("\nTable Head (10 rows):")
    # print(_df.head(10))
    print("\n\n")

## Helper Functions

In [ ]:
def header(msg: str) -> None:
    """
    Print a formatted header message.
    :param msg: Message to be printed as header.
    :return:
    """
    print("\n" + "=" * 50)
    print(msg)
    print("=" * 50)

In [ ]:
def clean_text(text: str) -> str:
    """
    Clean the input text by removing URLs, special characters, and extra spaces.
    :param text: Input text string.
    :return: Cleaned text string.
    """
    text = str(text).lower()
    text = re.sub(r"http\\S+|www\\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

## Exploratory Data Analysis

In [ ]:
train.shape, test.shape, sample_sub.shape

In [ ]:
train.sample(10)

In [ ]:
# Label distribution
print(train["label"].value_counts())
print(f"\nProportions:\n{train['label'].value_counts(normalize=True)}")

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=train, x="label")
plt.title("Label Distribution")
plt.show()

In [ ]:
train.describe().T

In [ ]:
corr = train.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
train["comment_length"] = train["comment"].str.len()
train["word_count"] = train["comment"].str.split().str.len()

header("TEXT STATISTICS BY LABEL")
print(train.groupby("label")[["comment_length", "word_count"]].mean())

## Feature Engineering

In [ ]:
def create_features(df):
    df = df.copy()

    # Text features
    df["comment_length"] = df["comment"].str.len()
    df["word_count"] = df["comment"].str.split().str.len()
    df["avg_word_length"] = df["comment"].apply(
        lambda x: np.mean([len(w) for w in str(x).split()]) if str(x).split() else 0
    )

    # Special characters
    df["exclamation_count"] = df["comment"].str.count("!")
    df["question_count"] = df["comment"].str.count("\\?")
    df["capital_ratio"] = df["comment"].apply(
        lambda x: (
            sum(1 for c in str(x) if c.isupper()) / len(str(x))
            if len(str(x)) > 0
            else 0
        )
    )

    # Engagement features
    df["total_votes"] = df["upvote"] + df["downvote"]
    df["vote_ratio"] = np.where(
        df["total_votes"] > 0, df["upvote"] / df["total_votes"], 0.5
    )
    df["vote_diff"] = df["upvote"] - df["downvote"]
    df["log_upvote"] = np.log1p(df["upvote"])
    df["log_downvote"] = np.log1p(df["downvote"])

    # Temporal features
    df["created_date"] = pd.to_datetime(df["created_date"])
    df["hour"] = df["created_date"].dt.hour
    df["day_of_week"] = df["created_date"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 6)).astype(int)

    # Emoticon features
    df["total_emoticons"] = df["emoticon_1"] + df["emoticon_2"] + df["emoticon_3"]

    # Post-level aggregations
    post_stats = (
        df.groupby("post_id")
        .agg({"upvote": ["mean", "count"], "downvote": "mean"})
        .reset_index()
    )

    post_stats.columns = [
        "post_id",
        "post_avg_upvote",
        "post_count",
        "post_avg_downvote",
    ]

    df = df.merge(post_stats, on="post_id", how="left")
    df["upvote_vs_post"] = df["upvote"] - df["post_avg_upvote"]
    df["downvote_vs_post"] = df["downvote"] - df["post_avg_downvote"]

    return df

In [ ]:
train = create_features(train)
test = create_features(test)

In [ ]:
print(f"Total Features: {len(train.columns)}")

## Model Building

In [ ]:
train["comment_clean"] = train["comment"].apply(clean_text)
test["comment_clean"] = test["comment"].apply(clean_text)

In [ ]:
# TF-IDF
tfidf = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), min_df=3, max_df=0.9)

In [ ]:
tfidf_train = tfidf.fit_transform(train["comment_clean"])
tfidf_test = tfidf.transform(test["comment_clean"])

In [ ]:
feature_cols = train.select_dtypes(include="number").columns.tolist()
feature_cols.remove("label")

In [ ]:
X_tab = train[feature_cols].fillna(0)
X_tab_test = test[feature_cols].fillna(0)

In [ ]:
scaler = StandardScaler()
X_tab_scaled = scaler.fit_transform(X_tab)
X_tab_test_scaled = scaler.transform(X_tab_test)

In [ ]:
# Combine features
X_train = np.hstack([tfidf_train.toarray(), X_tab_scaled])
X_test = np.hstack([tfidf_test.toarray(), X_tab_test_scaled])
y_train = train["label"].values

print(f"Training shape: {X_train.shape}")

In [ ]:
# Cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
n_classes = train["label"].unique().shape[0]
xgb_preds = np.zeros(len(X_test))
lgb_preds = np.zeros(len(X_test))

n_splits = skf.get_n_splits()

In [ ]:
xgb_test_preds = np.zeros((X_test.shape[0], n_classes))
lgb_test_preds = np.zeros((X_test.shape[0], n_classes))

oof_preds = np.zeros((X_train.shape[0], n_classes))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"Fold {fold + 1}:")

    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    # XGBoost
    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        early_stopping_rounds=50,
        learning_rate=0.05,
        random_state=42,
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    xgb_val_probs = xgb.predict_proba(X_val)
    xgb_test_preds += xgb.predict_proba(X_test) / n_splits

    # LightGBM
    lgb = LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, verbose=-1
    )
    lgb.fit(X_tr, y_tr)
    lgb_val_probs = lgb.predict_proba(X_val)
    lgb_test_preds += lgb.predict_proba(X_test) / n_splits

    val_ensemble_probs = (0.5 * xgb_val_probs) + (0.5 * lgb_val_probs)
    val_ensemble_preds = np.argmax(val_ensemble_probs, axis=1)
    val_acc = accuracy_score(y_val, val_ensemble_preds)
    print(f"\tAccuracy: {val_acc:.4f}")

In [ ]:
# Ensemble
final_test_probs = 0.5 * xgb_test_preds + 0.5 * lgb_test_preds
final_preds = np.argmax(final_test_probs, axis=1)

submission = pd.DataFrame({"ID": test.index + 1, "label": final_preds})
submission.to_csv("submission.csv", index=False)

print("\nSubmission created!")
print(submission["label"].value_counts())